In [ ]:
make clean
make
make armstub


In [ ]:
mkdir mnt && mkdir mnt/fat32 && mkdir mnt/ext4
sudo mount /dev/sda1 mnt/fat32 && sudo mount /dev/sda2 mnt/ext4
#mnt/fat32 會有 kernel8.img  備份 原本的 kernel8.img 

sudo cp kernel8.img mnt/fat32/kernel8.img
sudo cp armstub-new.bin mnt/fat32/armstub-new.bin
sudo cp config.txt mnt/fat32/config.txt
sudo umount mnt/fat32 && sudo umount mnt/ext4

In [ ]:
### makefile 內容  
# assemble startup
aarch64-linux-gnu-as -mcpu=cortex-a53 start.S -o start.o
# compile C
aarch64-linux-gnu-gcc -mcpu=cortex-a53 -ffreestanding -nostdlib -O2 -c main.c -o main.o
# link
aarch64-linux-gnu-ld -T linker.ld start.o main.o -o kernel.elf
# objcopy
aarch64-linux-gnu-objcopy kernel.elf -O binary kernel8.img


In [ ]:
ls /dev/tty*
# 應該會看到 ttyusb0   
正確流程是先把rpi 插頭拔掉  
在斷電狀態下  
sudo screen /dev/ttyUSB0 115200
插上rpi 插頭  就可以看到 boot code 
# MESS:00:00:02.362362:0: Device tree loaded to 
# Rasperry PI Bare Metal OS Initializing...
# Blame...
# Blame...
# Blame...
ctrl+A
k
# sudo screen /dev/ttyUSB0 115200,cs8,-ixon,-ixoff,-crtscts
# 好像要等很久才會看到 login   
# 要記得把 cmdline.txt 的 quiet 那些刪掉  才能看到很多 很酷的log



In [ ]:
sudo picocom -b 115200 /dev/ttyUSB0
Ctrl + A
Ctrl + X

#### 加上 armstub 之後 exception level 成功變成 3 
- 需要注意的是 我把 linker.ld 的 .= 0x80000  刪掉了 
  - 因為 armstub 好像是 從  addr 0 開始

T OK!

Rasperry PI Bare Metal OS Initializing...

Exception Level: 3


#### 在 boot.S 設定 general reg 之後 exception level 成功變成 1
不過 makefile 需要改動 COPS  
同時 .S file 也要用 `$(ARMGNU)-gcc `

```
MESS:00:00:02.462472:0: gpioman: gpioman_get_pin_num: pin SDCARD_CONTROL_POWE�
UART OK!

Rasperry PI 3B+ Bare Metal OS Initializing gogo...

Exception Level: 1
�
Terminating...
Thanks for using picocom
```

#### 設定 interrupt 打字的時候會 interrupt  
- 這樣就不用while loop receive 了
MESS:00:00:02.449749:0: uart: Baud rate change done...
MESS:00:00:02.455839:0: gpioman: gpioman_get_pin_num: pin SDCARD_CONTROL_POWE�
UART OK!

Rasperry PI 3B+ Bare Metal OS Initializing gogo...

Exception Level: 1
UART Recv: g
UART Recv: g
UART Recv: s
UART Recv: d
UART Recv: f
UART Recv: s
UART Recv: d
UART Recv: f
UART Recv: w
UART Recv: e
UART Recv: f
UART Recv: w
UART Recv: e
UART Recv: f


### interrupt timer 這樣子 timer3 速度是 timer1  4 倍速
MESS:00:00:02.451102:0: uart: Baud rate change done...
MESS:00:00:02.457195:0: gpioman: gpioman_get_pin_num: pin SDCARD_CONTROL_POWE�
UART OK!

Rasperry PI 3B+ Bare Metal OS Initializing gogo...

Exception Level: 1
Timer 3 received.
Timer 3 received.
Timer 3 received.
Timer 1 received.
Timer 3 received.
Timer 3 received.
Timer 3 received.
Timer 3 received.
Timer 1 received.
Timer 3 received.



### sleep 
UART OK!

Rasperry PI 3B+ Bare Metal OS Initializing gogo...

Exception Level: 1
sleeping 200 ms...sleeping 200 ms...sleeping 200 ms...sleeping 2 seconds...Timer 1 received.
Timer 1 received.
sleeping 2 seconds...Timer 1 received.
Timer 1 received.
sleeping 5 seconds...Timer 1 received.
Timer 1 received.
Timer 1 received.
Timer 1 received.
Timer 1 received.
Done!
Timer 1 received.
Timer 1 received.
Timer 1 received.

Terminating...
Thanks for using picocom